In [14]:
!pip install ultralytics -q



In [16]:
import os
import shutil
import uuid

# =========================
# PATHS
# =========================
bus_path = "/kaggle/input/datasets/seifallahahmed/buses-man/Bus.v1i.yolo26"
split1_src = "/kaggle/input/datasets/seifallahahmed/final-dataset/FINAL_DATASET3.0/FINAL_DATASET3.0/split_1"
split1_dst = "/kaggle/working/split_1_updated"

# =========================
# STEP 1: COPY SPLIT_1
# =========================
if not os.path.exists(split1_dst):
    shutil.copytree(split1_src, split1_dst)

print("✅ split_1 copied to working directory")

# =========================
# FUNCTION: ADD BUS DATA
# =========================
def add_bus_data(source_split, target_split):
    img_src = os.path.join(bus_path, source_split, "images")
    lbl_src = os.path.join(bus_path, source_split, "labels")

    img_dst = os.path.join(split1_dst, target_split, "images")
    lbl_dst = os.path.join(split1_dst, target_split, "labels")

    # 🔥 FIX: ensure folders exist
    os.makedirs(img_dst, exist_ok=True)
    os.makedirs(lbl_dst, exist_ok=True)

    if not os.path.exists(lbl_src):
        print(f"⚠️ Missing labels in {source_split}")
        return

    for file in os.listdir(lbl_src):
        label_path = os.path.join(lbl_src, file)

        with open(label_path, "r") as f:
            lines = f.readlines()

        new_lines = []

        for line in lines:
            parts = line.strip().split()
            parts[0] = "4"  # bus class
            new_lines.append(" ".join(parts))

        if len(new_lines) == 0:
            continue

        uid = str(uuid.uuid4())

        # save label
        with open(os.path.join(lbl_dst, uid + ".txt"), "w") as f:
            f.write("\n".join(new_lines))

        # copy image safely
        base = file.replace(".txt", "")
        found = False

        for ext in [".jpg", ".png", ".jpeg"]:
            src_img = os.path.join(img_src, base + ext)
            if os.path.exists(src_img):
                shutil.copy(src_img, os.path.join(img_dst, uid + ext))
                found = True
                break

        if not found:
            print(f"⚠️ Image not found for {file}")

# =========================
# STEP 2: ADD DATA
# =========================
add_bus_data("train", "train")
add_bus_data("valid", "valid")
add_bus_data("test", "train")  # optional boost

print("🔥 Bus data successfully added to split_1")

✅ split_1 copied to working directory
🔥 Bus data successfully added to split_1


In [17]:
import os
import yaml

base_path = "/kaggle/input/datasets/seifallahahmed/final-dataset/FINAL_DATASET3.0/FINAL_DATASET3.0"

splits = ["split_1", "split_2", "split_3"]

yaml_paths = []

for split in splits:
    data = {
        "train": f"{base_path}/{split}/train/images",
        "val": f"{base_path}/{split}/valid/images",
        "test": f"{base_path}/test/images",
        "nc": 6,
        "names": ['ambulance','firetruck','car','bus','motorcycle','truck']
    }

    yaml_path = f"/kaggle/working/{split}.yaml"
    
    with open(yaml_path, "w") as f:
        yaml.dump(data, f)

    yaml_paths.append(yaml_path)

    print(f"✅ Created {yaml_path}")

✅ Created /kaggle/working/split_1.yaml
✅ Created /kaggle/working/split_2.yaml
✅ Created /kaggle/working/split_3.yaml


In [ ]:
from ultralytics import YOLO

for i, yaml_file in enumerate(yaml_paths, 1):
    print(f"\n Training Split {i}")

    model = YOLO("yolov8s.pt")

    model.train(
        data=yaml_file,
        epochs=50,
        imgsz=640,
        batch=16,
        patience=10,
        name=f"model_split_{i}"
    )


🔥 Training Split 1
Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/split_1.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=model_split_15, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, over

In [ ]:
import shutil

# Zip the folder
shutil.make_archive('/kaggle/working/model_split_1', 'zip', '/kaggle/working/runs/detect/model_split_1')

# Download the zipped folder
from google.colab import files
files.download('/kaggle/working/model_split_1.zip')
